# **Phase 1 - Data Preprocessing & EDA**

In [1]:
%pip install -qU pandas pyarrow
%pip install -q scikit-learn numpy matplotlib seaborn tqdm huggingface_hub


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!ls workspace/data/raw

corpus.csv  train.csv


In [ ]:
# !hf download YuITC/vietnam-legal-documents --include "raw/*" --local-dir /workspace/data --repo-type dataset

In [3]:
import re
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [4]:
RAW_DATA_DIR     = Path('workspace/data/raw')
CLEANED_DATA_DIR = Path('workspace/data/cleaned')
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

corpus = pd.read_csv(RAW_DATA_DIR / 'corpus.csv')
train  = pd.read_csv(RAW_DATA_DIR / 'train.csv')

## **1. Data Overview**

In [5]:
print('Number of rows:')
print(f'- corpus: {len(corpus):,}')
print(f'- train : {len(train):,}')

Number of rows:
- corpus: 261,597
- train : 119,456


In [6]:
print('Data types:')
print(f'- corpus: {corpus.dtypes.to_dict()}')
print(f'- train : {train.dtypes.to_dict()}')

Data types:
- corpus: {'text': <StringDtype(na_value=nan)>, 'cid': dtype('int64')}
- train : {'question': <StringDtype(na_value=nan)>, 'context': <StringDtype(na_value=nan)>, 'cid': <StringDtype(na_value=nan)>, 'qid': dtype('int64')}


In [7]:
print('Sample corpus:')
display(corpus.head(2))

print('Sample train:')
display(train.head(2))

Sample corpus:


,text,cid
0,"Thông tư này hướng dẫn tuần tra, canh gác bảo ...",0
1,"1. Hàng năm trước mùa mưa, lũ, Ủy ban nhân dân...",1


Sample train:


,question,context,cid,qid
0,Người học ngành quản lý khai thác công trình t...,"['Khả năng học tập, nâng cao trình độ\n- Khối ...",[62492],161615
1,Nội dung lồng ghép vấn đề bình đẳng giới trong...,['Nội dung lồng ghép vấn đề bình đẳng giới tro...,[151154],80037


## **2. Corpus Data**

In [8]:
corpus.info()

<class 'pandas.DataFrame'>
RangeIndex: 261597 entries, 0 to 261596
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    261597 non-null  str  
 1   cid     261597 non-null  int64
dtypes: int64(1), str(1)
memory usage: 364.9 MB


In [9]:
corpus['word_count'] = corpus['text'].str.split().str.len()
corpus['word_count'].describe(percentiles=[.05, .25, .5, .75, .95, .99])

count    261597.000000
mean        240.138266
std         297.361953
min           1.000000
5%           47.000000
25%          96.000000
50%         175.000000
75%         307.000000
95%         626.000000
99%         896.040000
max       55368.000000
Name: word_count, dtype: float64

In [10]:
q_low  = corpus['word_count'].quantile(0.05)
q_high = corpus['word_count'].quantile(0.95)

corpus = corpus[
    (corpus['word_count'] >= q_low) &
    (corpus['word_count'] <= q_high)
]

print(corpus[['word_count']].describe())

          word_count
count  236199.000000
mean      216.251504
std       144.942626
min        47.000000
25%       102.000000
50%       175.000000
75%       287.000000
max       626.000000


## **3. Train Data**

In [11]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 119456 entries, 0 to 119455
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   question  119456 non-null  str  
 1   context   119456 non-null  str  
 2   cid       119456 non-null  str  
 3   qid       119456 non-null  int64
dtypes: int64(1), str(3)
memory usage: 178.5 MB


In [12]:
train['word_count'] = train['question'].str.split().str.len()
train['word_count'].describe(percentiles=[.05, .25, .5, .75, .95, .99])

count    119456.00000
mean         20.26208
std           6.53403
min           3.00000
5%           11.00000
25%          15.00000
50%          20.00000
75%          24.00000
95%          32.00000
99%          37.00000
max          60.00000
Name: word_count, dtype: float64

In [13]:
mapping = dict(zip(corpus['cid'], corpus['text']))

train['cid']     = train['cid'].apply(lambda x: list(map(int, re.findall(r'\d+', x))))
train['context'] = train['cid'].apply(lambda cid_list: [ctx for cid in cid_list if (ctx := mapping.get(cid)) is not None])

cid_len     = train['cid'].apply(len)
context_len = train['context'].apply(len)

train = train[
    (context_len == cid_len) &
    (context_len > 0) &
    (context_len <= 3)
]

In [14]:
train['n_relevant'] = train['cid'].apply(len)
train['n_relevant'].value_counts().sort_index()

n_relevant
1    97570
2     8945
3      655
Name: count, dtype: int64

In [15]:
corpus_cids = set(corpus['cid'].tolist())
train_cids  = set(cid for cids in train['cid'] for cid in cids)

print(f'Unique cids in corpus: {len(corpus_cids):,}')
print(f'Unique cids in train : {len(train_cids):,}')
print(f'Missing cids         : {len(train_cids - corpus_cids)}')
print(f'Corpus coverage      : {len(train_cids) / len(corpus_cids)*100:.1f}%')

Unique cids in corpus: 236,199
Unique cids in train : 62,364
Missing cids         : 0
Corpus coverage      : 26.4%


## **4. Save Data**

In [16]:
train_split, val_split = train_test_split(
    train,
    test_size    = 0.10,
    random_state = 42,
    stratify     = train['n_relevant']
)

train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print(f'Train split: {len(train_split):,}')
print(f'Val split  : {len(val_split):,}')

Train split: 96,453
Val split  : 10,717


In [17]:
train_split = train_split[['question', 'context', 'cid', 'qid']]
val_split   = val_split[['question', 'context', 'cid', 'qid']]
corpus      = corpus[['text', 'cid']]

train_split.to_parquet(CLEANED_DATA_DIR / 'train_split.parquet', index=False)
val_split.to_parquet(CLEANED_DATA_DIR / 'val_split.parquet', index=False)
corpus.to_parquet(CLEANED_DATA_DIR / 'corpus.parquet', index=False)

In [18]:
train_split.info()

<class 'pandas.DataFrame'>
RangeIndex: 96453 entries, 0 to 96452
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  96453 non-null  str   
 1   context   96453 non-null  object
 2   cid       96453 non-null  object
 3   qid       96453 non-null  int64 
dtypes: int64(1), object(2), str(1)
memory usage: 14.2+ MB


In [19]:
corpus.info()

<class 'pandas.DataFrame'>
Index: 236199 entries, 1 to 261596
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    236199 non-null  str  
 1   cid     236199 non-null  int64
dtypes: int64(1), str(1)
memory usage: 298.9 MB


In [21]:
!ls workspace/data/cleaned

corpus.parquet	train_split.parquet  val_split.parquet
